In [12]:
# Init

from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
VALIDATION_SIZE = 0.2
N_REPEATS = 10

DATA_DIR = Path("datasets/titanic")
OUTPUT_DIR = Path("outputs/titanic")

NUMERIC_FEATURES = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "HasCabin",
]
CATEGORICAL_FEATURES = [
    "Pclass",
    "Sex"
    "Embarked",
    "Title"
]

In [4]:
# Load

def load_titanic_data(data_dir):
    data_dir.mkdir(parents=True, exist_ok=True)

    base_url = (
        "https://raw.githubusercontent.com/"
        "ageron/handson-ml2/master/datasets/titanic/"
    )

    for filename in ("train.csv", "test.csv"):
        path = data_dir / filename

        if not path.is_file():
            urlretrieve(base_url + filename, str(path))

    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")

    return train_df, test_df

train_df, test_df = load_titanic_data(DATA_DIR)

print(train_df.shape)
print(test_df.shape)

train_df.head()

(891, 12)
(418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Data splitting

X_full = train_df.drop(columns="Survived")
y_full = train_df["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=VALIDATION_SIZE, stratify=y_full, random_state=RANDOM_STATE
)

print(X_train.shape)
print(X_val.shape)

(712, 11)
(179, 11)


In [6]:
# First Analyze

X_train.info()

summary = pd.DataFrame({
    "dtype": X_train.dtypes.astype(str),
    "missing": X_train.isnull().sum(),
    "missing_fraction": X_train.isna().mean(),
    "unique_values": X_train.nunique(),
})

summary

<class 'pandas.DataFrame'>
Index: 712 entries, 692 to 507
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Pclass       712 non-null    int64  
 2   Name         712 non-null    str    
 3   Sex          712 non-null    str    
 4   Age          575 non-null    float64
 5   SibSp        712 non-null    int64  
 6   Parch        712 non-null    int64  
 7   Ticket       712 non-null    str    
 8   Fare         712 non-null    float64
 9   Cabin        160 non-null    str    
 10  Embarked     710 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 66.8 KB


,dtype,missing,missing_fraction,unique_values
PassengerId,int64,0,0.000000,712
Pclass,int64,0,0.000000,3
Name,str,0,0.000000,712
Sex,str,0,0.000000,2
Age,float64,137,0.192416,85
SibSp,int64,0,0.000000,7
Parch,int64,0,0.000000,7
Ticket,str,0,0.000000,571
Fare,float64,0,0.000000,226
Cabin,str,552,0.775281,127


In [7]:
# More Splitting

target_distribution = pd.DataFrame({
    "count": y_train.value_counts(),
    "fraction": y_train.value_counts(normalize=True),
}).sort_index()

target_distribution

,count,fraction
Survived,,
0,439,0.616573
1,273,0.383427


In [9]:
# Features

def add_features(df):
    res = df.copy()

    title = (
        res["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
    )

    common_titles = ["Mr", "Miss", "Mrs", "Master"]

    res["Title"] = title.where(title.isin(common_titles), "Rare")
    res["HasCabin"] = (res["Cabin"].notna().astype(int))

    return res

add_features(X_train)[
    ["Name", "Title", "Cabin", "HasCabin"]
].head()

,Name,Title,Cabin,HasCabin
692,"Lam, Mr. Ali",Mr,NaN,0
481,"Frost, Mr. Anthony Wood 'Archie'",Mr,NaN,0
527,"Farthing, Mr. John",Mr,C95,1
855,"Aks, Mrs. Sam (Leah Rosen)",Mrs,NaN,0
801,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",Mrs,NaN,0


In [10]:
# Pipeline

def build_model(classifier):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )

    return Pipeline(
        ("features", FunctionTransformer(add_features, validate=False)),
        ("preprocessing", preprocessor),
        ("classifier", clone(classifier)),
    )

In [11]:
# CV

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

cv_splits = list(cv.split(X_train, y_train))
print(len(cv_splits))

50


In [ ]:
classifiers = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "svc": SVC(random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE),
}

